# 04 — Évaluation
Mesures chiffrées, figures et exemples visuels.

In [ ]:
# Installation (Google Colab / Kaggle avec GPU)
!git clone https://github.com/doaemalak/VLMs-l-gers-pour-l-gender-ou-juger-des-images.git
%cd VLMs-l-gers-pour-l-gender-ou-juger-des-images
!pip install -q -r requirements.txt

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
table = pd.read_csv('results/tableau_comparatif.csv')
table

## Mémoire GPU et score selon la précision

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, g in table.groupby('model'):
    axes[0].plot(g['precision'], g['gpu_peak_gb'], marker='o', label=name)
    axes[1].plot(g['precision'], g['CIDEr'], marker='o', label=name)
axes[0].set_ylabel('Mémoire GPU max (Go)'); axes[1].set_ylabel('CIDEr')
axes[0].axhline(12, ls='--', c='red', label='limite 12 Go')
for ax in axes: ax.legend()
plt.savefig('results/memoire_score.png', bbox_inches='tight'); plt.show()

## Exemples visuels : légendes des 6 configurations sur la même image

In [ ]:
from src.utils import load_image
subset = pd.read_csv('data/subset/subset.csv')
tags = [f'{m}_{p}' for m in ['vlm1', 'vlm2'] for p in ['normal', '8bit', '4bit']]
preds = {t: pd.read_csv(f'results/captions/{t}.csv').set_index('image') for t in tags}

for _, r in subset.head(3).iterrows():
    plt.imshow(load_image(r['path'])); plt.axis('off'); plt.show()
    print('Référence :', r['ref1'])
    for t in tags:
        print(f'{t:12s}:', preds[t].loc[r['image'], 'caption'])

## Option B — Corrélation entre le jugement du VLM et celui d'humains
Préparer `results/human_scores.csv` avec les colonnes `path, human_score` (une moyenne par image).

In [ ]:
from src.models import load_vlm, unload
from src.judge import judge_images
from src.metrics import correlation
from src.utils import load_config

human = pd.read_csv('results/human_scores.csv')
cfg = load_config('configs/vlm1.yaml')
rows = []
for precision in ['normal', '8bit', '4bit']:
    model, proc = load_vlm(cfg['model_id'], precision)
    j = judge_images(model, proc, human['path'].tolist(), cfg)
    unload(model)
    rows.append({'precision': precision, **correlation(j['score'], human['human_score'])})
pd.DataFrame(rows)